# 02 - SQL Aggregation & Before/After Consolidation Analysis
Reproduces the campaign-type aggregation in SQL and compares average open rate before vs. after the May 2026 sponsor list consolidation.

In [ ]:
import pandas as pd, sqlite3

df = pd.read_csv('../data/cleaned/0_Full_Cleaned_Dataset.csv')
df['Date Sent'] = pd.to_datetime(df['Date Sent'])

conn = sqlite3.connect(':memory:')
df.to_sql('edm_campaigns', conn, index=False)

In [ ]:
query = '''
SELECT Campaign_Type,
       COUNT(*) AS campaigns,
       ROUND(AVG("Open Rate"), 2) AS avg_open,
       ROUND(AVG(CTOR), 2) AS avg_ctor
FROM edm_campaigns
GROUP BY Campaign_Type
ORDER BY avg_ctor DESC;
'''
pd.read_sql(query, conn)

## Before / after May 2026 list consolidation

In [ ]:
df['Period'] = df['Date Sent'].apply(lambda d: 'Before' if d < pd.Timestamp('2026-05-01') else 'After')
before_after = df[df['Date Sent'] >= pd.Timestamp('2026-01-01')].groupby('Period').agg(
    avg_open=('Open Rate', 'mean'),
    campaign_count=('Email Subject', 'count')
).round(2)
before_after